In [ ]:
import pygame
import csv
import time
import random
import os
import json

# --- CONFIGURATION ---
SCREEN_WIDTH = 800
SCREEN_HEIGHT = 600
CURSOR_RADIUS = 20
TARGET_RADIUS = 30
FPS = 60
FILE_NAME = "experiment_data.csv"
RECORDING_DIR = "recordings"
NUM_ITERATIONS = 20
NUM_PRACTICE_ITERATIONS = 5

# Colors
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
RED   = (255, 0, 0)
BLUE  = (0, 0, 255)
GREEN = (0, 255, 0)
GRAY  = (128, 128, 128)
LIGHT_GRAY = (220, 220, 220)

# Game Modes
PRACTICE    = "practice"
INDIVIDUAL  = "individual"
COOPERATIVE = "cooperative"
PLAYBACK    = "playback"
AI          = "ai"

AI_CONTROLS_VERTICAL   = 1
AI_CONTROLS_HORIZONTAL = 0

PRESET_TARGETS = [
    [150, 100], [650, 100], [400, 150], [200, 250], [600, 250],
    [100, 350], [700, 350], [750, 80],  [550, 420], [300, 450],
    [450, 80],  [550, 100], [200, 200], [650, 200], [120, 480],
    [150, 400], [700, 450], [250, 500], [600, 480], [350, 520]
]


class ExperimentGame:
    def __init__(self, mode=INDIVIDUAL, recording_file=None, iteration=1, ai_axis=None,
                 target_pos=None, playback_axis=None, num_iterations=NUM_ITERATIONS,
                 participant_ids=None):
        pygame.init()
        self.screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
        pygame.display.set_caption("CogSci Joint Action Task")
        self.clock = pygame.time.Clock()
        self.running = True
        self.mode = mode
        self.iteration = iteration
        self.num_iterations = num_iterations
        self.target_hit = False
        self.participant_ids = participant_ids or []

        self.cursor_pos = [SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2]
        self.target_pos = target_pos if target_pos is not None else PRESET_TARGETS[iteration - 1]

        self.human_v   = [0, 0]
        self.partner_v = [0, 0]

        self.data_log   = []
        self.start_time = time.time()

        self.playback_frames = {'horizontal': [], 'vertical': []}
        self.playback_index  = 0
        self.playback_axis   = playback_axis
        if mode == PLAYBACK and recording_file:
            self.load_recording(recording_file)

        self.ai_control_axis = ai_axis if ai_axis is not None else random.randint(0, 1)

    def load_recording(self, filename):
        try:
            with open(filename, 'r') as f:
                data = json.load(f)
        except FileNotFoundError:
            print(f"Recording file {filename} not found.")
            return

        iterations = data.get('iterations', [])
        if not iterations:
            print("Recording has no iterations.")
            return

        idx = self.iteration - 1
        if idx >= len(iterations):
            print(f"Recording has no data for iteration {self.iteration}.")
            return

        iter_data = iterations[idx]
        self.playback_frames['horizontal'] = iter_data.get('horizontal', [])
        self.playback_frames['vertical']   = iter_data.get('vertical',   [])
        self.target_pos = iter_data.get('target_pos', self.target_pos)
        print(f"Loaded iteration {self.iteration}: "
              f"{len(self.playback_frames['horizontal'])} horizontal frames, "
              f"{len(self.playback_frames['vertical'])} vertical frames, "
              f"target={self.target_pos}")

    def get_playback_input(self):
        frames = self.playback_frames.get(self.playback_axis, [])
        if self.playback_index < len(frames):
            v = frames[self.playback_index]
            self.playback_index += 1
            return [v, 0] if self.playback_axis == 'horizontal' else [0, v]
        return [0, 0]

    def get_ai_input(self):
        dx = self.target_pos[0] - self.cursor_pos[0]
        dy = self.target_pos[1] - self.cursor_pos[1]
        k  = 0.035
        if self.ai_control_axis == AI_CONTROLS_VERTICAL:
            return [0, dy * k]
        else:
            return [dx * k, 0]

    def log_frame(self):
        elapsed = time.time() - self.start_time
        self.data_log.append([
            elapsed,
            self.cursor_pos[0], self.cursor_pos[1],
            self.human_v[0],    self.human_v[1],
            self.partner_v[0],  self.partner_v[1],
            self.target_pos[0], self.target_pos[1]
        ])

    def save_data(self):
        keys = ["timestamp", "cursor_x", "cursor_y", "h_vx", "h_vy", "p_vx", "p_vy", "target_x", "target_y"]
        if not os.path.exists(FILE_NAME):
            with open(FILE_NAME, "w", newline="") as f:
                csv.writer(f).writerow(keys)
        with open(FILE_NAME, "a", newline="") as f:
            csv.writer(f).writerows(self.data_log)
        print(f"Data saved to {FILE_NAME}")

    def draw_text(self, text, font, color, surface, x, y):
        surface.blit(font.render(text, True, color), (x, y))

    def run(self):
        font = pygame.font.Font(None, 24)

        while self.running and not self.target_hit:
            self.screen.fill(WHITE)

            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    self.running = False
                if event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE:
                    self.running = False

            keys = pygame.key.get_pressed()
            self.human_v = [0, 0]
            speed = 5

            if self.mode in (PRACTICE, INDIVIDUAL):
                if keys[pygame.K_LEFT]:  self.human_v[0] = -speed
                if keys[pygame.K_RIGHT]: self.human_v[0] =  speed
                if keys[pygame.K_UP]:    self.human_v[1] = -speed
                if keys[pygame.K_DOWN]:  self.human_v[1] =  speed
            elif self.mode == COOPERATIVE:
                if keys[pygame.K_LEFT]:  self.human_v[0] = -speed
                if keys[pygame.K_RIGHT]: self.human_v[0] =  speed
            elif self.mode == PLAYBACK:
                if self.playback_axis == 'horizontal':
                    if keys[pygame.K_UP]:   self.human_v[1] = -speed
                    if keys[pygame.K_DOWN]: self.human_v[1] =  speed
                else:
                    if keys[pygame.K_LEFT]:  self.human_v[0] = -speed
                    if keys[pygame.K_RIGHT]: self.human_v[0] =  speed
            elif self.mode == AI:
                if self.ai_control_axis == AI_CONTROLS_VERTICAL:
                    if keys[pygame.K_LEFT]:  self.human_v[0] = -speed
                    if keys[pygame.K_RIGHT]: self.human_v[0] =  speed
                else:
                    if keys[pygame.K_w]: self.human_v[1] = -speed
                    if keys[pygame.K_s]: self.human_v[1] =  speed

            self.partner_v = [0, 0]
            if self.mode == COOPERATIVE:
                if keys[pygame.K_w]: self.partner_v[1] = -speed
                if keys[pygame.K_s]: self.partner_v[1] =  speed
            elif self.mode == PLAYBACK:
                self.partner_v = self.get_playback_input()
            elif self.mode == AI:
                self.partner_v = self.get_ai_input()

            self.cursor_pos[0] += self.human_v[0] + self.partner_v[0]
            self.cursor_pos[1] += self.human_v[1] + self.partner_v[1]
            self.cursor_pos[0] = max(CURSOR_RADIUS, min(SCREEN_WIDTH  - CURSOR_RADIUS, self.cursor_pos[0]))
            self.cursor_pos[1] = max(CURSOR_RADIUS, min(SCREEN_HEIGHT - CURSOR_RADIUS, self.cursor_pos[1]))

            dist = ((self.cursor_pos[0] - self.target_pos[0])**2 +
                    (self.cursor_pos[1] - self.target_pos[1])**2)**0.5
            if dist < TARGET_RADIUS:
                self.target_hit = True
                print(f"Target Hit! Iteration {self.iteration} complete.")

            pygame.draw.circle(self.screen, RED,  self.target_pos, TARGET_RADIUS)
            pygame.draw.circle(self.screen, BLUE,
                               (int(self.cursor_pos[0]), int(self.cursor_pos[1])), CURSOR_RADIUS)

            self.draw_text(
                f"Mode: {self.mode.upper()} | Iteration: {self.iteration}/{self.num_iterations}",
                font, BLACK, self.screen, 10, 10)
            if self.mode in (PRACTICE, INDIVIDUAL):
                self.draw_text("Controls: Arrow Keys (all directions)", font, BLACK, self.screen, 10, 35)
            elif self.mode == COOPERATIVE:
                self.draw_text("P1: LEFT/RIGHT  |  P2: W/S", font, BLACK, self.screen, 10, 35)
            elif self.mode == PLAYBACK:
                if self.playback_axis == 'horizontal':
                    self.draw_text("Playback: LEFT/RIGHT  |  You: UP/DOWN", font, BLACK, self.screen, 10, 35)
                else:
                    self.draw_text("Playback: UP/DOWN  |  You: LEFT/RIGHT", font, BLACK, self.screen, 10, 35)
            elif self.mode == AI:
                ai_txt  = "UP/DOWN" if self.ai_control_axis == AI_CONTROLS_VERTICAL else "LEFT/RIGHT"
                you_txt = "LEFT/RIGHT" if self.ai_control_axis == AI_CONTROLS_VERTICAL else "UP/DOWN (W/S)"
                self.draw_text(f"AI: {ai_txt}  |  You: {you_txt}", font, BLACK, self.screen, 10, 35)
            self.draw_text("ESC to quit", font, GRAY, self.screen, 10, 60)

            self.log_frame()
            pygame.display.flip()
            self.clock.tick(FPS)

        if self.target_hit:
            self.save_data()


def show_id_input_screen(mode):
    """Show participant ID input before the session. Returns list of IDs, or None if cancelled."""
    pygame.init()
    screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
    pygame.display.set_caption("Participant ID")
    clock      = pygame.time.Clock()
    font       = pygame.font.Font(None, 36)
    small_font = pygame.font.Font(None, 24)

    if mode == COOPERATIVE:
        labels = ["Player 1 ID:", "Player 2 ID:"]
    else:
        labels = ["Participant ID:"]

    values       = [""] * len(labels)
    active_field = 0

    while True:
        screen.fill(WHITE)

        title = font.render("Enter Participant ID", True, BLACK)
        screen.blit(title, (SCREEN_WIDTH // 2 - title.get_width() // 2, 70))

        sub = small_font.render("Use TAB or ENTER to move between fields. Press ENTER on the last field to continue.", True, GRAY)
        screen.blit(sub, (SCREEN_WIDTH // 2 - sub.get_width() // 2, 120))

        for i, (label, value) in enumerate(zip(labels, values)):
            y = 200 + i * 110

            label_surf = font.render(label, True, BLACK)
            screen.blit(label_surf, (150, y))

            box_rect  = pygame.Rect(150, y + 42, 500, 48)
            box_color = BLUE if i == active_field else GRAY
            pygame.draw.rect(screen, LIGHT_GRAY, box_rect)
            pygame.draw.rect(screen, box_color, box_rect, 3)

            display_text = value + ("|" if i == active_field else "")
            screen.blit(font.render(display_text, True, BLACK), (box_rect.x + 10, box_rect.y + 8))

        hint = small_font.render("ESC to cancel", True, GRAY)
        screen.blit(hint, (SCREEN_WIDTH // 2 - hint.get_width() // 2, 520))

        pygame.display.flip()

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                return None
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    pygame.quit()
                    return None
                elif event.key == pygame.K_TAB:
                    active_field = (active_field + 1) % len(labels)
                elif event.key == pygame.K_RETURN:
                    if active_field < len(labels) - 1:
                        active_field += 1        # move to next field
                    else:
                        pygame.quit()
                        return values            # all fields filled — done
                elif event.key == pygame.K_BACKSPACE:
                    values[active_field] = values[active_field][:-1]
                elif event.unicode.isprintable():
                    values[active_field] += event.unicode

        clock.tick(FPS)


def save_session_recording(session_records, filename):
    if not os.path.exists(RECORDING_DIR):
        os.makedirs(RECORDING_DIR)
    data = {
        'mode': INDIVIDUAL,
        'num_iterations': len(session_records),
        'timestamp': int(time.time()),
        'iterations': session_records
    }
    with open(filename, 'w') as f:
        json.dump(data, f, indent=2)
    print(f"Recording saved: {filename} ({len(session_records)} iterations)")


def show_menu():
    pygame.init()
    screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
    pygame.display.set_caption("CogSci Joint Action Task")
    clock      = pygame.time.Clock()
    font       = pygame.font.Font(None, 36)
    small_font = pygame.font.Font(None, 24)

    selected = 0
    modes = [
        (PRACTICE,    "0. Practice Mode (5 trials, warm-up)"),
        (INDIVIDUAL,  "1. Individual Mode (You control alone)"),
        (COOPERATIVE, "2. Cooperative Mode (2 players)"),
        (PLAYBACK,    "3. Playback Mode (Play with your own recording)"),
        (AI,          "4. AI Mode (Play against AI agent)")
    ]

    while True:
        screen.fill(WHITE)
        title = font.render("Select Game Mode", True, BLACK)
        screen.blit(title, (SCREEN_WIDTH // 2 - title.get_width() // 2, 50))
        for i, (_, label) in enumerate(modes):
            color  = BLUE if i == selected else BLACK
            prefix = ">>> " if i == selected else "    "
            screen.blit(small_font.render(prefix + label, True, color), (50, 130 + i * 55))
        screen.blit(small_font.render("UP/DOWN to select, ENTER to confirm, ESC to exit", True, GRAY),
                    (SCREEN_WIDTH // 2 - 200, 520))
        pygame.display.flip()

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                return None
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_UP:    selected = (selected - 1) % len(modes)
                if event.key == pygame.K_DOWN:  selected = (selected + 1) % len(modes)
                if event.key == pygame.K_RETURN:
                    pygame.quit()
                    return modes[selected][0]
                if event.key == pygame.K_ESCAPE:
                    pygame.quit()
                    return None
        clock.tick(FPS)


def show_recording_menu():
    pygame.init()
    screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
    pygame.display.set_caption("Select Recording")
    clock      = pygame.time.Clock()
    font       = pygame.font.Font(None, 36)
    small_font = pygame.font.Font(None, 24)

    recordings = []
    if os.path.exists(RECORDING_DIR):
        recordings = sorted([f for f in os.listdir(RECORDING_DIR) if f.endswith('.json')], reverse=True)

    if not recordings:
        screen.fill(WHITE)
        screen.blit(small_font.render("No recordings found. Run INDIVIDUAL mode first.", True, BLACK),
                    (50, SCREEN_HEIGHT // 2))
        pygame.display.flip()
        pygame.time.wait(2000)
        pygame.quit()
        return None

    selected = 0
    while True:
        screen.fill(WHITE)
        screen.blit(font.render("Select Recording", True, BLACK), (SCREEN_WIDTH // 2 - 100, 50))
        for i, rec in enumerate(recordings):
            color  = BLUE if i == selected else BLACK
            prefix = ">>> " if i == selected else "    "
            screen.blit(small_font.render(prefix + rec, True, color), (50, 150 + i * 40))
        screen.blit(small_font.render("UP/DOWN to select, ENTER to confirm, ESC to cancel", True, GRAY),
                    (50, 500))
        pygame.display.flip()

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                return None
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_UP:    selected = (selected - 1) % len(recordings)
                if event.key == pygame.K_DOWN:  selected = (selected + 1) % len(recordings)
                if event.key == pygame.K_RETURN:
                    pygame.quit()
                    return os.path.join(RECORDING_DIR, recordings[selected])
                if event.key == pygame.K_ESCAPE:
                    pygame.quit()
                    return None
        clock.tick(FPS)


def show_iteration_ready_screen(mode, iteration, total, ai_axis=None, playback_axis=None):
    pygame.init()
    screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
    pygame.display.set_caption("Ready")
    clock      = pygame.time.Clock()
    font       = pygame.font.Font(None, 36)
    small_font = pygame.font.Font(None, 24)

    while True:
        screen.fill(WHITE)
        screen.blit(font.render(f"Iteration {iteration} of {total}", True, BLACK),
                    (SCREEN_WIDTH // 2 - 100, 50))
        screen.blit(small_font.render("Controls:", True, BLACK), (100, 140))

        y = 175
        if mode in (PRACTICE, INDIVIDUAL):
            screen.blit(small_font.render("Arrow Keys — control the ball in all directions", True, BLACK), (120, y))
        elif mode == COOPERATIVE:
            screen.blit(small_font.render("P1: LEFT/RIGHT arrows (horizontal)", True, BLACK), (120, y))
            screen.blit(small_font.render("P2: W/S keys (vertical)", True, BLACK), (120, y + 28))
        elif mode == PLAYBACK:
            if playback_axis == 'horizontal':
                screen.blit(small_font.render("Playback controls: LEFT/RIGHT (horizontal)", True, BLUE), (120, y))
                screen.blit(small_font.render("You control: UP/DOWN (vertical) with Arrow Keys", True, BLACK), (120, y + 28))
            else:
                screen.blit(small_font.render("Playback controls: UP/DOWN (vertical)", True, BLUE), (120, y))
                screen.blit(small_font.render("You control: LEFT/RIGHT (horizontal) with Arrow Keys", True, BLACK), (120, y + 28))
        elif mode == AI:
            ai_ctrl  = "UP/DOWN" if ai_axis == AI_CONTROLS_VERTICAL else "LEFT/RIGHT"
            you_ctrl = "LEFT/RIGHT arrows" if ai_axis == AI_CONTROLS_VERTICAL else "W/S keys"
            screen.blit(small_font.render(f"AI controls: {ai_ctrl}", True, BLUE), (120, y))
            screen.blit(small_font.render(f"You control: {you_ctrl}", True, BLACK), (120, y + 28))

        screen.blit(small_font.render("Goal: move the BLUE ball into the RED target", True, BLACK), (100, 290))
        screen.blit(small_font.render("Move the cursor exactly on the target as fast as", True, RED), (100, 325))
        screen.blit(small_font.render("possible, using the most direct path possible!", True, RED), (100, 350))

        screen.blit(font.render("Press SPACE to start", True, BLUE),
                    (SCREEN_WIDTH // 2 - 130, 415))
        screen.blit(small_font.render("ESC to return to menu", True, GRAY),
                    (SCREEN_WIDTH // 2 - 90, 480))
        pygame.display.flip()

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                return False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_SPACE:
                    pygame.quit()
                    return True
                if event.key == pygame.K_ESCAPE:
                    pygame.quit()
                    return False
        clock.tick(FPS)


if __name__ == "__main__":
    while True:
        mode = show_menu()
        if mode is None:
            print("Exiting.")
            break

        num_iterations = NUM_PRACTICE_ITERATIONS if mode == PRACTICE else NUM_ITERATIONS

        # Ask for participant ID(s) — skip for practice
        participant_ids = []
        if mode != PRACTICE:
            participant_ids = show_id_input_screen(mode)
            if participant_ids is None:
                print("ID entry cancelled. Returning to menu.")
                continue
            id_str = " | ".join(f"P{i+1}: {pid}" for i, pid in enumerate(participant_ids))
            print(f"Session started — {id_str}")

        recording_file = None
        if mode == PLAYBACK:
            recording_file = show_recording_menu()
            if recording_file is None:
                continue

        target_order = list(range(len(PRESET_TARGETS)))
        random.shuffle(target_order)

        individual_save_path = None
        if mode == INDIVIDUAL:
            if not os.path.exists(RECORDING_DIR):
                os.makedirs(RECORDING_DIR)
            pid = participant_ids[0] if participant_ids else "unknown"
            individual_save_path = os.path.join(
                RECORDING_DIR, f"session_{pid}_{int(time.time())}.json")
            print(f"Individual session will be saved to: {individual_save_path}")

        session_records = []

        for iteration in range(1, num_iterations + 1):
            ai_axis = random.randint(0, 1) if mode == AI else None

            playback_axis = None
            if mode == PLAYBACK:
                playback_axis = random.choice(['horizontal', 'vertical'])
                print(f"Iteration {iteration}: playback axis = {playback_axis}")

            target_pos = PRESET_TARGETS[target_order[iteration - 1]]

            if not show_iteration_ready_screen(mode, iteration, num_iterations,
                                               ai_axis=ai_axis, playback_axis=playback_axis):
                print("Cancelled. Returning to menu.")
                break

            game = ExperimentGame(
                mode=mode, recording_file=recording_file,
                iteration=iteration, ai_axis=ai_axis,
                target_pos=target_pos, playback_axis=playback_axis,
                num_iterations=num_iterations,
                participant_ids=participant_ids
            )
            game.run()

            if not game.target_hit:
                print("Iteration not completed. Returning to menu.")
                break

            if mode == INDIVIDUAL:
                session_records.append({
                    'iteration': iteration,
                    'target_pos': game.target_pos,
                    'horizontal': [entry[3] for entry in game.data_log],
                    'vertical':   [entry[4] for entry in game.data_log],
                })
                save_session_recording(session_records, individual_save_path)

        print(f"Done with {mode.upper()} mode. Returning to menu...")
